<a href="https://colab.research.google.com/github/Hion-cy/ClassFiles/blob/main/AL263158_Practica_SQL_Fintech_Vacio_DML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Materia:** Ingeniería de Datos Avanzada
###**Alumna:** Carmen Yolanda Hion Vela
###**Matrícula:** AL263158


# 📘 Práctica SQL – Fintech Databank

Este notebook contiene los **ejercicios prácticos** sobre la base de datos `fintech_databank.db`.  
Tu objetivo será escribir consultas SQL para resolver cada problema.

👉 **Recuerda:** conecta la base de datos antes de empezar:
```python
%sql sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
```


In [6]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The sql extension is already loaded. To reload it, use:
  %reload_ext sql



## 🔹 Instrucciones generales
1. Conéctate a la base de datos.  

In [7]:
%sql sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db


2. Antes de cada ejercicio, revisa las tablas disponibles con:
   ```sql
   %%sql
   SELECT name FROM sqlite_master WHERE type='table';
   ```



In [8]:
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


name
clientes
cuentas
transacciones
tipos_cambio


3. Ejecuta tus consultas SQL en las celdas de código marcadas como `%%sql`.  
4. Guarda tus respuestas en este notebook.

## Ejercicio 1. Exploración básica


- Lista todas las filas de la tabla **clientes**.  



In [20]:
%%sql
SELECT * FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C001,Ana Torres,CDMX,Retail
C002,Luis Gómez,Guadalajara,Retail
C003,Textiles J&R,Monterrey,PYME


- Muestra únicamente el nombre y la ciudad de cada cliente.  


In [18]:
%%sql
SELECT nombre, ciudad FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


nombre,ciudad
Ana Torres,CDMX
Luis Gómez,Guadalajara
Textiles J&R,Monterrey


- ¿Cuántos clientes hay en total?  

In [16]:
%%sql
SELECT COUNT(*) FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


COUNT(*)
3


## Ejercicio 2. Cuentas de clientes


- Obtén todas las cuentas del cliente **C001**.  



In [24]:
%%sql
SELECT * FROM cuentas WHERE cliente_id = 'C001';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cuenta_id,cliente_id,moneda,tipo,fecha_apertura
A1001,C001,MXN,Vista,2024-11-10
A1002,C001,USD,Ahorro,2024-12-01


- Lista las cuentas que están en moneda **USD**.  


In [25]:
%%sql
SELECT * FROM cuentas WHERE moneda = 'USD';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cuenta_id,cliente_id,moneda,tipo,fecha_apertura
A1002,C001,USD,Ahorro,2024-12-01


- Muestra el número de cuentas que tiene cada cliente (usa `GROUP BY`).  

In [27]:
%%sql
SELECT cliente_id, COUNT(*) AS num_cuentas
FROM cuentas
GROUP BY cliente_id;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,num_cuentas
C001,2
C002,1
C003,1


## Ejercicio 3. Transacciones


- Obtén todas las transacciones de la cuenta **A1001**.  


In [28]:
%%sql
SELECT * FROM transacciones WHERE cuenta_id = 'A1001';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0001,A1001,2025-07-01,Nómina,25000,Depósito
T0002,A1001,2025-07-03,Renta,-12000,Pago
T0003,A1001,2025-07-05,Super,-2200,Pago
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión


- Calcula el **saldo neto** de esa cuenta sumando la columna `monto`.  


In [30]:
%%sql
SELECT cuenta_id, SUM(monto) AS saldo_neto
FROM transacciones
WHERE cuenta_id = 'A1001';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cuenta_id,saldo_neto
A1001,10765


- Lista las transacciones que sean **comisiones**.  


In [45]:
%%sql
SELECT * FROM transacciones WHERE categoria = 'Comisión';



 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0006,A2001,2025-07-04,Comisión manejo,-45,Comisión
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión


In [40]:

%%sql
--Comisiones cuenta A1001
SELECT * FROM transacciones
WHERE cuenta_id = 'A1001' AND categoria = 'Comisión';


 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.
(sqlite3.OperationalError) near "%": syntax error
[SQL: %%sql
SELECT * FROM transacciones WHERE categoria = 'Comisión';]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


## Ejercicio 4. Reportes con agrupaciones


- Calcula el total de transacciones (suma de `monto`) por **categoría**.     


In [46]:
%%sql
SELECT categoria, SUM(monto) as monto_total
from transacciones
group by categoria;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


categoria,monto_total
Comisión,-80
Depósito,25800
Interés,123.5
Pago,-82200
Transferencia,95000



## Ejercicio 5. **Modificar datos (UPDATE)**

> **Objetivo:** actualizar valores existentes usando `WHERE` de forma segura.

1. Cambia el `segmento` del cliente **C002** de *Retail* a **PYME**.  



In [47]:

%%sql
SELECT * FROM clientes WHERE cliente_id = 'C002';


 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C002,Luis Gómez,Guadalajara,Retail


In [49]:
%%sql
UPDATE clientes
SET segmento = 'PYME'
WHERE cliente_id = 'C002';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
1 rows affected.


[]

In [50]:

%%sql
SELECT * FROM clientes WHERE cliente_id = 'C002';


 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C002,Luis Gómez,Guadalajara,PYME


In [51]:
%%sql
SELECT * FROM clientes;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C001,Ana Torres,CDMX,Retail
C002,Luis Gómez,Guadalajara,PYME
C003,Textiles J&R,Monterrey,PYME


2. Corrige la `ciudad` de **Textiles J&R (C003)** a **Saltillo**.  

In [52]:
%%sql
SELECT * FROM clientes WHERE cliente_id = 'C003';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C003,Textiles J&R,Monterrey,PYME


In [53]:
%%sql
UPDATE clientes
SET ciudad = 'Saltillo'
WHERE cliente_id = 'C003';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
1 rows affected.


[]

In [54]:
%%sql
SELECT * FROM clientes WHERE cliente_id = 'C003';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


cliente_id,nombre,ciudad,segmento
C003,Textiles J&R,Saltillo,PYME


> **Tip:** siempre valida primero con un `SELECT` y después aplica el `UPDATE`.


## Ejercicio 6. **Borrar datos (DELETE)**

> **Objetivo:** eliminar filas específicas manteniendo la integridad y usando `WHERE`.

1. Elimina **solo** las transacciones de la cuenta **A3001** del día **2025-07-12**.  



In [56]:
%%sql
SELECT * FROM transacciones
WHERE cuenta_id = 'A3001' AND fecha = '2025-07-12';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0008,A3001,2025-07-12,Transferencia cliente,95000,Transferencia


In [58]:
%%sql
DELETE FROM transacciones
WHERE cuenta_id = 'A3001' AND fecha = '2025-07-12';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
1 rows affected.


[]

In [59]:
%%sql
SELECT * FROM transacciones
WHERE cuenta_id = 'A3001' AND fecha = '2025-07-12';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria


In [60]:
%%sql
SELECT * FROM transacciones;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0001,A1001,2025-07-01,Nómina,25000,Depósito
T0002,A1001,2025-07-03,Renta,-12000,Pago
T0003,A1001,2025-07-05,Super,-2200,Pago
T0004,A1002,2025-07-07,Ahorro mensual,800,Depósito
T0005,A2001,2025-07-02,Interés mensual,120,Interés
T0006,A2001,2025-07-04,Comisión manejo,-45,Comisión
T0007,A3001,2025-07-10,Pago nómina empleados,-68000,Pago
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión
T0010,A1002,2025-07-20,Interés mensual,3.5,Interés


2. Elimina la **comisión** de `A2001` del **2025-07-04**.  


In [62]:
%%sql
SELECT * FROM transacciones
 WHERE cuenta_id = 'A2001' AND fecha = '2025-07-04';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0006,A2001,2025-07-04,Comisión manejo,-45,Comisión


In [63]:
%%sql
DELETE FROM transacciones
WHERE cuenta_id = 'A2001' AND fecha = '2025-07-04';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
1 rows affected.


[]

In [64]:
%%sql
SELECT * FROM transacciones
 WHERE cuenta_id = 'A2001' AND fecha = '2025-07-04';

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria


In [65]:
%%sql
SELECT * FROM transacciones;

 * sqlite:////content/drive/MyDrive/ClassFilesIDA/fintech_databank.db
Done.


tx_id,cuenta_id,fecha,concepto,monto,categoria
T0001,A1001,2025-07-01,Nómina,25000,Depósito
T0002,A1001,2025-07-03,Renta,-12000,Pago
T0003,A1001,2025-07-05,Super,-2200,Pago
T0004,A1002,2025-07-07,Ahorro mensual,800,Depósito
T0005,A2001,2025-07-02,Interés mensual,120,Interés
T0007,A3001,2025-07-10,Pago nómina empleados,-68000,Pago
T0009,A1001,2025-07-15,Comisión retiro,-35,Comisión
T0010,A1002,2025-07-20,Interés mensual,3.5,Interés


> **Tip:** prueba primero con `SELECT` para verificar el subconjunto a borrar.